# Ascend C算子开发的基本概念
前文我们已经围绕**算子与CANN架构**的关联完成阐述，本节将正式聚焦Ascend C算子开发核心，从**定义、开发场景、抽象硬件架构**三个维度，拆解开发必备的基础概念，为后续实操搭建完整的知识框架，本节学习大纲如下。
- 什么是Ascend C
- 什么时候需要开发Ascend C算子
- Ascend C对应的抽象硬件架构

---

## 1. 什么是Ascend C
Ascend C是CANN专为算子开发打造的编程语言，原生兼容C/C++标准规范，兼顾**开发效率**与**硬件运行性能**；基于Ascend C编写的算子程序，经编译器编译和运行时调度后，可直接在昇腾NPU上高效执行。

使用Ascend C，开发者可以基于昇腾AI硬件，开发者能基于昇腾NPU快速实现自定义创新算法，**使用Ascend C编程语言开发的算子称为Ascend C算子。**

<img src="./images/ascend_c_runtime.png" alt="ascend_c_runtime"  width="700px" >

Ascend C 提供**多层级API**体系，可灵活满足不同场景的算子开发诉求，各层级API定位与能力如下：

- **基础API**：基于Tensor进行编程的C++类库API，实现单指令级抽象，为底层算子开发提供灵活控制能力。

- **高阶API**：封装单核公共算法，涵盖一些常见的计算算法（如卷积、矩阵运算等），显著降低复杂算法开发门槛。

- **算子模板库**：基于模板提供算子完整实现参考，简化Tiling（切分算法）开发，支撑用户自定义扩展。

- **Python前端（PyAsc）**：PyAsc编程语言基于Python原生接口，提供芯片底层完备编程能力，支持基于Python接口开发高性能Ascend C算子。

<img src="./images/ascend_c_architecture.png" alt="ascend_c_architecture"  width="700px" >

#

> ✨ 本教程重点：聚焦基础&高阶 API展开讲解，指导开发者通过C++类库API完成高性能Ascend C算子开发。

#


## 2. 什么时候需要开发Ascend C算子
常规业务场景下，开发者可直接使用CANN提供的原生算子，无需自定义开发；**仅当遇到以下场景时，才需要考虑开发Ascend C算子**，以此解决算子兼容性问题或提升业务运行性能：

- **推理场景(算子不兼容)**：使用ATC工具将TensorFlow、Caffe、ONNX 等第三方框架模型，转换为昇腾平台离线模型时，遇到平台未支持的算子。

- **推理场景(业务逻辑加速)**：应用程序中的数学运算类逻辑（如查找最大值、数据类型转换、推理结果后处理等），可通过自定义算子实现，再通过ACL接口调用，利用昇腾NPU完成硬件加速。  
✅ 示例：分类应用中，需从模型推理结果中查找置信度最高的前 5 个标签，可自定义实现 ArgMax（最大值查找）算子，替代 CPU 端纯软件计算，大幅提升后处理效率。

- **训练场景（算子不兼容）**：将 TensorFlow、PyTorch 等第三方框架的网络训练脚本迁移到昇腾平台时，遇到平台未支持的算子。

- **性能调优（算子性能不足）**：网络调优过程中，发现某原生算子执行性能较低，可重新开发高性能 Ascend C 算子对其进行替换，优化整体网络的运行效率。

## 3. Ascend C对应的抽象硬件架构
AI Core 是昇腾 AI 处理器的核心计算单元，一枚昇腾 AI 处理器内部集成多个 AI Core；其原生硬件架构较为复杂，而 Ascend C 对 AI Core 的并行计算架构做了**轻量化抽象——屏蔽不同硬件型号的底层差异，简化硬件细节**，让开发者无需深入理解复杂的硬件原理，即可完成算子开发，大幅降低开发门槛。

<img src="./images/ai_core_abstract_architecture.png" alt="ai_core_abstract_architecture"  width="700px" >

### 3.1 核心硬件组件构成
Ascend C 抽象后的 AI Core 架构，核心包含**计算单元、存储单元、搬运单元**三大类组件，各组件分工明确、协同工作，完成数据的计算与处理，三类组件的具体构成及功能如下表所示：

<table style="float: left; border-collapse: collapse; margin: 0 10px 10px 0; font-size: 14px;">
<tr style="background-color:#f0f0f0">
<td align="center"><strong>组件分类</strong></td>
<td align="center"><strong>组件名称</strong></td>
<td align="center"><strong>组件功能</strong></td>
</tr>
<tr>
<td align="left" rowspan="3">计算单元</td>
<td align="left">Scalar</td>
<td align="left">执行地址计算、循环控制等标量计算工作，并把向量计算、矩阵计算、数据搬运、同步指令发射给对应单元执行。</td>
</tr>
<tr>
<td align="left">Vector</td>
<td align="left">负责执行向量运算。</td>
</tr>
<tr>
<td align="left">Cube</td>
<td align="left">负责执行矩阵运算。</td>
</tr>
<tr>
<td align="left">存储单元</td>
<td align="left">Local Memory</td>
<td align="left">AI Core的内部存储，所有芯片缓冲区的统称，对应数据类型为LocalTensor。</td>
</tr>
<tr>
<td align="left">搬运单元</td>
<td align="left">DMA（Direct Memory Access）</td>
<td align="left">负责数据搬运，包括Global Memory和Local Memory之间的数据搬运以及不同层级Local Memory之间的数据搬运。</td>
</tr>
</table>
<div style="clear: both;"></div>

> 💡 补充存储单元说明：
> - 外部存储：AI Core 可访问的 NPU 设备内存（Device 内存），称为 Global Memory，对应数据类型为 GlobalTensor；
> - 数据处理原则：AI Core 的计算单元仅能对 Local Memory 中的数据进行计算，需通过 DMA 完成 Global Memory 与 Local Memory 之间的数据交互。

### 3.2 核心工作过程（三大关键流）
开发者理解抽象硬件架构时，需重点掌握 AI Core 内部**异步指令流、同步信号流、计算数据流**三个核心过程，三者协同实现硬件的并行计算与数据处理（可对应架构图中蓝、绿、红三色箭头）：

- **异步指令流（蓝色）**：并行计算的核心

Scalar 计算单元读取指令序列，将向量计算、矩阵计算、数据搬运等指令分别发射到对应单元的专属指令队列；Vector、Cube、DMA 单元异步并行执行各自队列中的指令，充分利用硬件的并行计算能力。

- **同步信号流（绿色）**：保证执行逻辑正确性

不同指令队列的指令间可能存在依赖关系（如先搬入数据再执行计算），Scalar 计算单元会向对应单元下发同步指令，协调各单元的执行时序，确保指令按正确的逻辑关系运行。

- **计算数据流（红色）**：数据处理的基本流程

DMA 先将数据从 Global Memory 搬运到 Local Memory；Vector/Cube 计算单元对 Local Memory 中的数据完成计算，并将结果写回 Local Memory；最后 DMA 将处理完成的数据从 Local Memory 搬运回 Global Memory，完成一次完整的计算流程。

---

## 课后练习
本节总体介绍了CANN异构计算架构与昇腾NPU的硬件细节，请根据学习内容完成以下题目进行自测。

1. （判断题）AI Core的计算单元仅能对Global Memory中的数据进行计算。

2. （判断题）异步指令流负责协调各单元的执行时序，确保指令按正确的逻辑关系运行。

3. （单选题）在AI Core抽象架构中，负责执行向量运算的是？  
    A. Scalar   
    B. Vector   
    C. Cube   
    D. DMA   

4. （单选题）计算数据流的基本流程是？  
    A. DMA搬运数据 → 计算单元计算 → DMA搬运结果  
    B. 计算单元计算 → DMA搬运数据 → DMA搬运结果  
    C. DMA搬运结果 → 计算单元计算 → DMA搬运数据  
    D. 计算单元计算 → DMA搬运结果 → DMA搬运数据  

5. （多选题）AI Core的计算单元包括哪些？  
    A. Scalar  
    B. DMA  
    C. Cube  
    D. Vector  

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/01.04_answer.txt